# QF600 Asset Pricing — Backtesting a Portfolio Strategy & Computing Alpha

Backtest an **investor portfolio** against a **benchmark portfolio**, both rebalanced on a fixed
schedule, then decompose the result into the part explained by market exposure and the part that is not:

$$R_p - R_f \;=\; \alpha \;+\; \beta\,(R_m - R_f) \;+\; \varepsilon$$


Documentation in this [Github link](https://github.com/cyee02/MQF/tree/main/QF600/Assignment%201%20-%20Compute%20Alpha)

---
## 1. Set up

All third-party libraries are declared here and nowhere else in the notebook.

In [1258]:
# Environment bootstrap: install anything missing, then switch on an interactive table viewer.
import importlib
import subprocess
import sys


def ensure_installed(package: str, module: str | None = None) -> None:
    """pip-install `package` only if `module` cannot already be imported.

    Parameters
    ----------
    package : str         name to pass to `pip install`
    module  : str | None  import name, when it differs from `package` (e.g. "lets_plot"
                          for "lets-plot"); defaults to `package`

    Returns
    -------
    None  called for its side effect: the package is importable afterwards
    """
    try:
        importlib.import_module(module or package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


ensure_installed("yfinance")
ensure_installed("lets-plot", "lets_plot")
ensure_installed("tabulate")            # pandas needs it for DataFrame.to_markdown() in §8

try:
    import google.colab                                              # noqa: F401
    IN_COLAB = True
    get_ipython().run_line_magic("load_ext", "google.colab.data_table")  # paginated tables in Colab
except ImportError:
    IN_COLAB = False                                                 # local -> Data Wrangler plugin

print(f"Running in Colab: {IN_COLAB}")

Running in Colab: False


In [1259]:
import datetime as dt

import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats

from IPython.display import Markdown, display
from lets_plot import *
LetsPlot.setup_html()

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.6f}".format)
pd.set_option("display.max_rows", 10)

---
## 2. Parameters

Everything a user would want to change lives in this one cell.

The strategy horizon is expressed as a **start / end date** rather than a raw day count: the spec
fixes it at "01 Jan 2016", and pinning the dates avoids the calendar-days vs trading-days ambiguity.
The horizon *in trading days* is derived from the data further below.

`OPTIMISE_WEIGHTS` picks which of the two investor portfolios is backtested. Off, it is the fixed
`INVESTOR_PORTFOLIO` mix, held for the whole horizon. On, §4 re-solves the maximum-Sharpe mix at
every rebalance by Monte Carlo — `MC_RUNS` random long-only allocations, scored on the trailing
`LOOKBACK_DAYS` of returns. That window ends the day the block begins, so the weights a block trades
on are never fitted to the returns that block goes on to earn.

In [ ]:
INVESTOR_PORTFOLIO  = {
    "QQQ": 0.334,  # Tracks NASDAQ-100 Index
    "DBMF": 0.333,  #  Managed Futures Strategy. With reference to top 20 Commodity Trading Advisor (CTA) hedge funds, the fund buys a smaller, cheaper group of about 10 to 15 liquid futures and forward contracts to match the hedge funds' returns
    "GLD" : 0.333,  # SPDR Gold Shares ETF
}

BENCHMARK_PORTFOLIO = {
    "SPY" : 1,          # S&P 500 ETF
}

START_DATE       = "2016-01-01"               # ~10 years of history
END_DATE         = dt.date.today()
REBALANCE_DAYS   = 60                         # trading days between rebalances (~1 quarter)
# REBALANCE_DAYS = int((END_DATE - dt.date.fromisoformat(START_DATE)).days * 0.7)  # No rebalance for now, just compute the returns of the portfolio over the entire period

OPTIMISE_WEIGHTS = False                       # True  -> re-solve the investor's max-Sharpe mix at
                                              #          every rebalance (Markowitz MC, §4)
                                              # False -> hold INVESTOR_PORTFOLIO for the whole horizon
MC_RUNS          = 10_000                      # Monte Carlo draws per rebalance
LOOKBACK_DAYS    = 252 * 5                    # trailing window the optimiser estimates from (~5 year)
RANDOM_SEED      = 42                         # fixes the draws, so a re-run reproduces the backtest

INITIAL_CAPITAL  = 100_000                    # USD at inception
TRADING_DAYS     = 252                        # annualisation factor
CVAR_CONFIDENCE  = 0.95                       # CVaR tail cutoff: average of the worst 5% of days
RISK_FREE_TICKER = "^TNX"                     # CBOE 10-year Treasury yield index
# RISK_FREE_TICKER = "^FVX"                     # CBOE 5-year Treasury yield index

# The investor weights are the mix held throughout when OPTIMISE_WEIGHTS is off, and both the
# investable universe and the early-block fallback when it is on. Either way they must be valid.
for name, portfolio in [("Investor", INVESTOR_PORTFOLIO), ("Benchmark", BENCHMARK_PORTFOLIO)]:
    assert np.isclose(sum(portfolio.values()), 1.0), f"{name} weights must sum to 1.0"

TICKERS = list(dict.fromkeys([*INVESTOR_PORTFOLIO, *BENCHMARK_PORTFOLIO]))   # de-duplicated
TICKERS

['QQQ', 'DBMF', 'GLD', 'SPY']

---
## 3. Data

### 3.1 Daily close prices

The download is kept twice: `prices_raw` exactly as returned, so §3.2 can measure what is
missing, and `prices` restricted to dates every ticker traded, which is what the backtest uses.

In [1261]:
prices_raw =\
(
    yf.download(TICKERS,                # every ticker in one request
                start        = START_DATE,
                end          = END_DATE,
                auto_adjust  = True,    # adjust for splits & dividends -> total-return prices
                progress     = False
               )
    ["Close"]                           # daily close only
    [TICKERS]                           # restore the declared column order
)

prices =\
(
    prices_raw
    .dropna()                           # keep only dates every ticker traded
)

prices

Ticker,QQQ,DBMF,GLD,SPY
Date,,,,
2019-05-08,177.694687,16.375015,120.910004,258.351501
2019-05-09,176.738159,16.384186,121.199997,257.569763
2019-05-10,176.958191,16.386812,121.430000,258.863647
2019-05-13,170.817230,16.266966,122.669998,252.358322
2019-05-14,172.692062,16.359959,122.459999,254.640625
...,...,...,...,...
2026-09-01,707.640015,31.590000,396.750000,761.780029
2026-09-02,709.239990,31.500000,402.779999,765.159973
2026-09-03,717.669983,31.410000,410.220001,773.169983


### 3.2 Data completeness

Measured on `prices_raw`, before `.dropna()` hid anything. A ticker can be short of days for two
very different reasons, so they are counted separately:

- **a late start or early end** — the fund simply did not exist on those dates (`SOXX` and `GLD`
  both predate 2016, but this catches it for any ticker swapped into §2);
- **interior gaps** — a day inside the ticker's own coverage where every other ticker printed a
  price and this one did not. These are the ones worth worrying about.

Any date where even one ticker is missing is dropped from the backtest, so the shared calendar is
the intersection, not the union.

In [1262]:
calendar_days = len(prices_raw)          # union calendar: any date at least one ticker traded

gaps_within_coverage =\
(
    prices_raw
    .apply(lambda column: column
                          .loc[column.first_valid_index():column.last_valid_index()]
                          .isna()
                          .sum())      # missing days strictly inside each ticker's own history
)

completeness =\
(
    pd.DataFrame({"First Quote"  : prices_raw.apply(lambda column: column.first_valid_index()),
                  "Last Quote"   : prices_raw.apply(lambda column: column.last_valid_index()),
                  "Trading Days" : calendar_days,
                  "Observed"     : prices_raw.count(),
                  "Missing"      : prices_raw.isna().sum(),
                  "Interior Gaps": gaps_within_coverage})
    .assign(**{"Missing %": lambda frame: (frame["Missing"] / calendar_days).map("{:.2%}".format)})
    .rename_axis("Ticker")
)

print(f"Union calendar : {prices_raw.index[0]:%d %b %Y} -> {prices_raw.index[-1]:%d %b %Y} "
      f"({calendar_days:,} trading days, the union across all tickers)")
print(f"Shared calendar: {len(prices):,} days kept by .dropna() "
      f"({calendar_days - len(prices):,} dropped, {1 - len(prices) / calendar_days:.2%})")

if gaps_within_coverage.any():
    print("Interior gaps (a ticker missing a day the others traded): "
          + ", ".join(f"{ticker} {count}"
                      for ticker, count in gaps_within_coverage[gaps_within_coverage > 0].items()))
else:
    print("No interior gaps: every ticker quotes a price on every day inside its own coverage.")

completeness

Union calendar : 04 Jan 2016 -> 08 Sep 2026 (2,685 trading days, the union across all tickers)
Shared calendar: 1,844 days kept by .dropna() (841 dropped, 31.32%)
No interior gaps: every ticker quotes a price on every day inside its own coverage.


,First Quote,Last Quote,Trading Days,Observed,Missing,Interior Gaps,Missing %
Ticker,,,,,,,
QQQ,2016-01-04,2026-09-08,2685,2685,0,0,0.00%
DBMF,2019-05-08,2026-09-08,2685,1844,841,0,31.32%
GLD,2016-01-04,2026-09-08,2685,2685,0,0,0.00%
SPY,2016-01-04,2026-09-08,2685,2685,0,0,0.00%


### 3.3 Risk-free rate

`^TNX` quotes the 10-year Treasury yield as an annualised **percent** (e.g. `2.27` = 2.27%). It is
converted to a compounded daily rate and aligned to the trading calendar of `prices`, so it can be
subtracted from daily returns directly.

In [1263]:
rf_daily =\
(
    yf.download(RISK_FREE_TICKER,
                start        = START_DATE,
                end          = END_DATE,
                auto_adjust  = True,
                progress     = False
               )
    ["Close"]
    .squeeze()                          # single-column frame -> Series
    .div(100)                           # percent -> decimal, annualised
    .add(1)
    .pow(1 / TRADING_DAYS)              # annual -> daily, compounded
    .sub(1)
    .reindex(prices.index)              # onto the equity trading calendar
    .ffill()                            # carry the last quote over yield-market holidays
    .bfill()                            # cover a missing first observation
    .rename("rf_daily")
)

rf_daily.to_frame()

,rf_daily
Date,
2019-05-08,0.000097
2019-05-09,0.000096
2019-05-10,0.000096
2019-05-13,0.000094
2019-05-14,0.000095
...,...
2026-09-01,0.000186
2026-09-02,0.000186
2026-09-03,0.000185


### 3.4 Daily asset returns and the derived horizon

Day 0 is inception, so its return is forced to `0` rather than dropped — this keeps every curve
starting at exactly `INITIAL_CAPITAL` on the first date.

In [1264]:
asset_returns =\
(
    prices
    .pct_change()                       # daily simple returns
    .fillna(0)                          # day 0 = inception, no return yet
)

HORIZON_DAYS = len(asset_returns)       # horizon, in trading days

print(f"Horizon: {prices.index[0]:%d %b %Y} -> {prices.index[-1]:%d %b %Y} "
      f"({HORIZON_DAYS:,} trading days, {HORIZON_DAYS / TRADING_DAYS:.2f} years)")
print(f"Number of Rebalances over the horizon: {(HORIZON_DAYS - 1) // REBALANCE_DAYS}")

asset_returns

Horizon: 08 May 2019 -> 08 Sep 2026 (1,844 trading days, 7.32 years)
Number of Rebalances over the horizon: 30


Ticker,QQQ,DBMF,GLD,SPY
Date,,,,
2019-05-08,0.000000,0.000000,0.000000,0.000000
2019-05-09,-0.005383,0.000560,0.002398,-0.003026
2019-05-10,0.001245,0.000160,0.001898,0.005023
2019-05-13,-0.034703,-0.007314,0.010212,-0.025130
2019-05-14,0.010976,0.005717,-0.001712,0.009044
...,...,...,...,...
2026-09-01,-0.012724,0.006692,-0.028574,-0.006870
2026-09-02,0.002261,-0.002849,0.015198,0.004437
2026-09-03,0.011886,-0.002857,0.018472,0.010468


---
## 4. Reusable functions

Seven small building blocks, each doing one job: rebalance a portfolio, solve for the max-Sharpe
weights on a window, schedule those weights across the rebalance blocks, measure a drawdown,
measure tail risk, summarise risk & return, and regress one excess-return series on another.

In [1265]:
w      = pd.Series(INVESTOR_PORTFOLIO, dtype = float)
w
blocks = pd.Series(np.arange(len(asset_returns)) // REBALANCE_DAYS,
                    index = asset_returns.index)

value_per_dollar =\
(
    asset_returns
    [w.index]                        # only this portfolio's assets, in weight order
    .add(1)
    .groupby(blocks)
    # .cumprod()                       # growth of $1 per asset, restarting each block
    # .mul(w, axis = "columns")        # weighted at the block's target weights
    # .sum(axis = "columns")           # -> portfolio value per $1 invested at block start
)
value_per_dollar

In [1266]:
def do_rebalance(asset_returns: pd.DataFrame,
                 weights: dict[str, float] | pd.DataFrame,
                 initial_capital: float,
                 rebalance_days: int) -> pd.Series:
    """Compound a portfolio daily, snapping back to target `weights` every `rebalance_days`.

    Weights are assumed perfectly divisible: no share counts, no transaction costs.

    Between rebalances each holding drifts with its own return; on a rebalance date the
    portfolio value is redistributed across the target weights and compounding resumes.
    Vectorised by splitting the horizon into fixed-length blocks, so there is no day loop.

    Parameters
    ----------
    asset_returns   : pd.DataFrame      daily simple returns; DatetimeIndex rows x ticker
                                        columns, first row 0.0 (inception)
    weights         : dict | DataFrame  either one target weight per ticker, held for the whole
                                        horizon, or per-block weights as returned by
                                        `build_block_weights` — one row per block, ticker
                                        columns. Tickers must be a subset of
                                        `asset_returns.columns`, and each row must sum to 1.0
    initial_capital : float             portfolio value on the first date, in dollars
    rebalance_days  : int               trading days between rebalances

    Returns
    -------
    pd.Series  float, named "Value", indexed like `asset_returns`: the portfolio's dollar
               value on each date, starting at `initial_capital`
    """
    blocks = pd.Series(np.arange(len(asset_returns)) // rebalance_days,
                       index = asset_returns.index)

    # One weight vector per block. A dict is the constant-weight case, broadcast to every block,
    # so both regimes take exactly the same path from here on.
    weights_by_block =\
    (
        pd.DataFrame([weights] * (blocks.iloc[-1] + 1), dtype = float)
        if isinstance(weights, dict) else
        weights.astype(float)
    )

    value_per_dollar =\
    (
        asset_returns
        [weights_by_block.columns]       # only this portfolio's assets, in weight order
        .add(1)
        .groupby(blocks)
        .cumprod()                       # growth of $1 per asset, restarting each block
        .mul(weights_by_block            # at each block's own target weights
             .loc[blocks.to_numpy()]
             .set_axis(asset_returns.index))
        .sum(axis = "columns")           # -> portfolio value per $1 invested at block start
    )

    capital_at_block_start =\
    (
        value_per_dollar
        .groupby(blocks)
        .last()                          # each block's total growth factor
        .shift(1, fill_value = 1.0)      # block N starts with what blocks 0..N-1 earned
        .cumprod()
        .mul(initial_capital)
    )

    return (
        value_per_dollar
        .mul(blocks.map(capital_at_block_start))
        .rename("Value")
    )

In [1267]:
def optimise_weights(window_returns: pd.DataFrame,
                     rf_window: pd.Series,
                     trading_days: int,
                     mc_runs: int,
                     rng: np.random.Generator) -> np.ndarray:
    """Pick the long-only mix with the highest Sharpe ratio, by random search.

    This is the Markowitz tangency portfolio — the best return per unit of risk available
    from these assets — approximated by sampling instead of solved in closed form.

    Three steps:

    1. Draw `mc_runs` random weight vectors. Each is non-negative and rescaled to sum to 1,
       so every candidate is a valid long-only, fully-invested portfolio by construction —
       nothing has to be rejected or penalised.
    2. Score them all in one matrix product. `(days x assets) @ (assets x runs)` produces one
       excess-return path per candidate, and two column-wise reductions turn those paths into
       annualised Sharpe ratios. There is no loop over draws.
    3. Return the weights of the best-scoring draw.

    Step 2 uses the same Sharpe formula as `compute_metrics` (`ddof = 1`, annualised by
    `sqrt(trading_days)`), so the quantity maximised here is the quantity section 6.1 reports.

    Being a sample rather than a solve, the answer is approximate: more assets need more draws,
    and the search is biased toward balanced mixes because normalising uniform draws clusters
    them near equal weight rather than spreading them evenly over the simplex.

    Parameters
    ----------
    window_returns : pd.DataFrame        daily simple returns over the estimation window;
                                         date rows x ticker columns
    rf_window      : pd.Series           daily risk-free rate, sharing that index
    trading_days   : int                 annualisation factor (252)
    mc_runs        : int                 number of random allocations to draw
    rng            : np.random.Generator seeded source, so the search is reproducible

    Returns
    -------
    np.ndarray  one weight per column of `window_returns`, in that order; non-negative and
                summing to 1.0

    See Also
    --------
    README.md, section 2 — a worked example with input and output.
    """
    draws  = rng.random((mc_runs, window_returns.shape[1]))
    draws /= draws.sum(axis = 1, keepdims = True)      # long-only and fully invested

    # (days x assets) @ (assets x runs) -> one excess-return series per candidate allocation
    excess =\
    (
        window_returns
        .sub(rf_window, axis = "index")
        .to_numpy()
        @ draws.T
    )

    # Same annualised Sharpe as `compute_metrics`, so the objective maximised here is the
    # number §6.1 reports. ddof = 1 matches pandas' default.
    with np.errstate(invalid = "ignore", divide = "ignore"):
        sharpe = excess.mean(axis = 0) / excess.std(axis = 0, ddof = 1) * np.sqrt(trading_days)

    return draws[np.nanargmax(sharpe)]


def build_block_weights(asset_returns: pd.DataFrame,
                        rf_daily: pd.Series,
                        weights: dict[str, float],
                        rebalance_days: int,
                        optimise: bool,
                        lookback_days: int,
                        trading_days: int,
                        mc_runs: int,
                        rng: np.random.Generator,
                        min_window: int = 20) -> pd.DataFrame:
    """Build the table of weights each rebalance block trades on — one row per block.

    This is the schedule `do_rebalance` consumes. It has two modes:

    - `optimise = False` — every row is the section 2 mix, so the fixed-weight backtest is
      reproduced exactly. Flipping the flag off always gets the original result back.
    - `optimise = True`  — each block is solved separately by `optimise_weights`, on the
      returns that came immediately before it.

    The estimation window for block `b` is the `lookback_days` of returns ending where the
    block starts:

        rows [b * rebalance_days - lookback_days  :  b * rebalance_days]

    That right edge is the important part. The slice stops at the block's own first row, so a
    block is only ever allocated using returns that had already happened when the rebalance was
    placed — it never sees the returns it goes on to earn. Fitting on the full sample instead
    would hand every block the answer key and inflate alpha by construction.

    Two kinds of block fall back to the section 2 mix instead of being optimised: block 0,
    whose window is empty, and any early block whose window is still shorter than `min_window`.
    Note the window only reaches its full `lookback_days` once `b * rebalance_days` exceeds it;
    until then a block estimates on whatever shorter history exists, which is noisier.

    One `rng` is threaded through every block, so a single seed fixes the whole schedule.

    Parameters
    ----------
    asset_returns  : pd.DataFrame        daily simple returns; DatetimeIndex rows x ticker columns
    rf_daily       : pd.Series           daily risk-free rate, sharing that index
    weights        : dict[str, float]    the section 2 target mix — its keys are the investable
                                         universe, its values the fallback allocation
    rebalance_days : int                 trading days between rebalances
    optimise       : bool                `OPTIMISE_WEIGHTS` from section 2
    lookback_days  : int                 longest trailing estimation window
    trading_days   : int                 annualisation factor (252)
    mc_runs        : int                 Monte Carlo draws per rebalance
    rng            : np.random.Generator seeded source, so the whole schedule is reproducible
    min_window     : int                 shortest window worth estimating from

    Returns
    -------
    pd.DataFrame  indexed 0 .. n_blocks-1 by block number, columns ordered like `weights`;
                  every row non-negative and summing to 1.0

    See Also
    --------
    README.md, section 2 — a worked example with input and output.
    """
    tickers  = list(weights)
    n_blocks = -(-len(asset_returns) // rebalance_days)        # ceiling division

    block_weights =\
    (
        pd.DataFrame([weights] * n_blocks, dtype = float)
        .rename_axis("Block")
    )

    if not optimise:
        return block_weights

    for block in range(n_blocks):
        start  = block * rebalance_days                        # first row the block trades
        window = asset_returns[tickers].iloc[max(0, start - lookback_days) : start]

        if len(window) >= min_window:
            block_weights.iloc[block] = optimise_weights(window,
                                                         rf_daily.loc[window.index],
                                                         trading_days,
                                                         mc_runs,
                                                         rng)

    return block_weights

In [1268]:
def compute_drawdown(value: pd.Series) -> pd.Series:
    """Percentage decline from the running peak, for every date.

    Parameters
    ----------
    value : pd.Series  a positive level series, e.g. the dollar values from `do_rebalance`

    Returns
    -------
    pd.Series  float, indexed like `value`: 0.0 on a day that sets a new peak, negative
               below it (-0.25 = 25% under water)
    """
    return (
        value
        .div(value.cummax())
        .sub(1)
    )


def compute_cvar(portfolio_returns: pd.Series,
                 confidence: float = 0.95) -> float:
    """Conditional value at risk: the average daily return on the worst `1 - confidence` days.

    Historical (non-parametric): no distribution is assumed, the sample's own left tail *is*
    the estimate. Value at risk is the quantile that cuts the tail off; CVaR (expected
    shortfall) is the mean of everything beyond it, so it answers "when a bad day happens,
    how bad on average" rather than "how bad at the threshold" — and unlike VaR it is
    sensitive to how fat the tail is behind the cutoff.

    Parameters
    ----------
    portfolio_returns : pd.Series  daily simple returns of one portfolio
    confidence        : float      tail cutoff, 0.95 -> the worst 5% of days

    Returns
    -------
    float  negative decimal, same sign convention as a drawdown (-0.03 = the worst 5% of
           days lose 3% on average)
    """
    value_at_risk = portfolio_returns.quantile(1 - confidence)   # left-tail cutoff

    return (
        portfolio_returns
        [portfolio_returns <= value_at_risk]                     # the tail itself
        .mean()
    )

In [1269]:
def compute_metrics(portfolio_returns: pd.Series,
                    rf_daily: pd.Series,
                    trading_days: int) -> dict[str, float]:
    """Return / volatility / Sharpe, reported both over the full horizon and annualised.

    Parameters
    ----------
    portfolio_returns : pd.Series  daily simple returns of one portfolio
    rf_daily          : pd.Series  daily risk-free rate, sharing the same index
    trading_days      : int        annualisation factor (252)

    Returns
    -------
    dict[str, float]  six metrics keyed by name: "Horizon Return", "Annualised Return",
                      "Horizon Volatility", "Annualised Volatility", "Horizon Sharpe",
                      "Annualised Sharpe". Returns and volatilities are decimals
                      (0.12 = 12%); the Sharpe ratios are unitless.
    """
    horizon_days   = len(portfolio_returns)
    horizon_return = portfolio_returns.add(1).prod() - 1
    daily_vol      = portfolio_returns.std()

    excess_returns = portfolio_returns.sub(rf_daily)
    annual_sharpe  = excess_returns.mean() / excess_returns.std() * np.sqrt(trading_days)

    return {
        "Horizon Return"       : horizon_return,
        "Annualised Return"    : (1 + horizon_return) ** (trading_days / horizon_days) - 1,
        "Horizon Volatility"   : daily_vol * np.sqrt(horizon_days),
        "Annualised Volatility": daily_vol * np.sqrt(trading_days),
        "Horizon Sharpe"       : annual_sharpe * np.sqrt(horizon_days / trading_days),
        "Annualised Sharpe"    : annual_sharpe,
    }

In [1270]:
def compute_alpha_beta(excess_investor: pd.Series,
                       excess_benchmark: pd.Series,
                       trading_days: int,
                       confidence: float = 0.95) -> dict[str, float]:
    """OLS of the investor's excess return on the benchmark's excess return, with inference.

        R_p - R_f = alpha + beta * (R_m - R_f) + epsilon

    beta  -> sensitivity to the benchmark (just the market)
    alpha -> average excess return the benchmark does not explain (skill)

    Tests H_0: alpha = 0 two ways. The textbook standard error assumes i.i.d. homoskedastic
    residuals; daily returns are neither, being volatility-clustered and mildly autocorrelated,
    so a Newey-West HAC standard error is reported alongside it. Where the two disagree, trust
    the HAC one -- the textbook error is the optimistic one.

    Parameters
    ----------
    excess_investor  : pd.Series  daily investor excess returns  (R_p - R_f), the regressand
    excess_benchmark : pd.Series  daily benchmark excess returns (R_m - R_f), the regressor;
                                  same index as `excess_investor`
    trading_days     : int        annualisation factor (252)
    confidence       : float      two-sided confidence level for the alpha interval (0.95)

    Returns
    -------
    dict[str, float]  "Beta" (unitless slope), "Alpha (daily)" (decimal intercept),
                      "Annualised Alpha" (decimal, alpha compounded over `trading_days`),
                      "R-squared" (0.0 to 1.0), then the inference on alpha:
                      "SE Alpha (OLS)" / "t-stat (OLS)" / "p-value (OLS)" under i.i.d. errors,
                      "SE Alpha (HAC)" / "t-stat (HAC)" / "p-value (HAC)" Newey-West corrected,
                      "NW Lags" (Bartlett truncation lag), and the annualised alpha interval
                      "Annual Alpha CI Low" / "Annual Alpha CI High" (decimals, from the HAC SE)
    """
    beta        = excess_investor.cov(excess_benchmark) / excess_benchmark.var()
    alpha_daily = excess_investor.mean() - beta * excess_benchmark.mean()

    # --- residuals and the textbook (i.i.d.) standard error of the intercept ---
    residuals = excess_investor - alpha_daily - beta * excess_benchmark
    n         = len(excess_investor)
    dof       = n - 2                                    # two estimated parameters
    sigma2    = (residuals ** 2).sum() / dof
    x_mean    = excess_benchmark.mean()
    sxx       = ((excess_benchmark - x_mean) ** 2).sum()

    se_ols    = np.sqrt(sigma2 * (1 / n + x_mean ** 2 / sxx))

    # --- Newey-West HAC standard error: sandwich with Bartlett-weighted autocovariances ---
    design    = np.column_stack([np.ones(n), excess_benchmark.to_numpy()])
    scores    = residuals.to_numpy()[:, None] * design   # h_t = e_t * X_t
    nw_lags   = int(4 * (n / 100) ** (2 / 9))            # standard truncation rule

    meat = scores.T @ scores
    for lag in range(1, nw_lags + 1):
        gamma = scores[lag:].T @ scores[:-lag]
        meat += (1 - lag / (nw_lags + 1)) * (gamma + gamma.T)

    bread     = np.linalg.inv(design.T @ design)
    se_hac    = np.sqrt((bread @ meat @ bread)[0, 0])    # [0, 0] is the intercept's variance

    t_ols     = alpha_daily / se_ols
    t_hac     = alpha_daily / se_hac
    t_crit    = stats.t.ppf(0.5 + confidence / 2, dof)

    # Compounding is monotone, so transforming the daily endpoints preserves the coverage.
    ci_daily  = alpha_daily - t_crit * se_hac, alpha_daily + t_crit * se_hac

    return {
        "Beta"                : beta,
        "Alpha (daily)"       : alpha_daily,
        "Annualised Alpha"    : (1 + alpha_daily) ** trading_days - 1,
        "R-squared"           : excess_investor.corr(excess_benchmark) ** 2,
        "SE Alpha (OLS)"      : se_ols,
        "t-stat (OLS)"        : t_ols,
        "p-value (OLS)"       : 2 * stats.t.sf(abs(t_ols), dof),
        "SE Alpha (HAC)"      : se_hac,
        "t-stat (HAC)"        : t_hac,
        "p-value (HAC)"       : 2 * stats.t.sf(abs(t_hac), dof),
        "NW Lags"             : nw_lags,
        "Annual Alpha CI Low" : (1 + ci_daily[0]) ** trading_days - 1,
        "Annual Alpha CI High": (1 + ci_daily[1]) ** trading_days - 1,
    }

---
## 5. Backtest

### 5.1 Portfolio values

Both portfolios start at `INITIAL_CAPITAL` and compound daily, rebalancing every
`REBALANCE_DAYS`.

The benchmark holds its §2 mix throughout. The investor's mix comes from `build_block_weights` —
identical to the §2 mix when `OPTIMISE_WEIGHTS` is off, and re-solved for maximum Sharpe on the
preceding `LOOKBACK_DAYS` at every rebalance when it is on.

In [1271]:
# One weight vector per rebalance block. With OPTIMISE_WEIGHTS off this is the §2 mix repeated,
# so the fixed-weight backtest is reproduced exactly; with it on, each block is solved on its own
# trailing window. The benchmark is always static — it is the yardstick.
investor_weights =\
(
    build_block_weights(asset_returns,
                        rf_daily,
                        INVESTOR_PORTFOLIO,
                        REBALANCE_DAYS,
                        OPTIMISE_WEIGHTS,
                        LOOKBACK_DAYS,
                        TRADING_DAYS,
                        MC_RUNS,
                        np.random.default_rng(RANDOM_SEED))
)

portfolio_values =\
(
    pd.DataFrame(
        {"Investor" : do_rebalance(asset_returns, investor_weights,
                                   INITIAL_CAPITAL, REBALANCE_DAYS),
         "Benchmark": do_rebalance(asset_returns, BENCHMARK_PORTFOLIO,
                                   INITIAL_CAPITAL, REBALANCE_DAYS)}
    )
    .rename_axis("Date")
)

portfolio_values

,Investor,Benchmark
Date,,
2019-05-08,"100,000.000000","100,000.000000"
2019-05-09,"99,918.724862","99,697.412913"
2019-05-10,"100,028.769263","100,198.236121"
2019-05-13,"98,972.288563","97,680.222763"
2019-05-14,"99,455.960017","98,563.632708"
...,...,...
2026-09-01,"254,481.076231","294,861.854867"
2026-09-02,"256,201.255006","296,170.128219"
2026-09-03,"258,946.925671","299,270.559113"


### 5.2 Portfolio returns and excess returns

Rebalancing is value-neutral (it only reshuffles an unchanged total), so the daily return of the
portfolio is simply the day-over-day change in its value, including across rebalance dates.

In [1272]:
portfolio_returns =\
(
    portfolio_values
    .pct_change()
    .fillna(0)                           # day 0 = inception
)

portfolio_returns

,Investor,Benchmark
Date,,
2019-05-08,0.000000,0.000000
2019-05-09,-0.000813,-0.003026
2019-05-10,0.001101,0.005023
2019-05-13,-0.010562,-0.025130
2019-05-14,0.004887,0.009044
...,...,...
2026-09-01,-0.014640,-0.006870
2026-09-02,0.006760,0.004437
2026-09-03,0.010717,0.010468


In [1273]:
excess_returns =\
(
    portfolio_returns
    .sub(rf_daily, axis = "index")       # R - R_f, the input to Sharpe and to alpha/beta
)

excess_returns

,Investor,Benchmark
Date,,
2019-05-08,-0.000097,-0.000097
2019-05-09,-0.000909,-0.003122
2019-05-10,0.001005,0.004927
2019-05-13,-0.010656,-0.025225
2019-05-14,0.004792,0.008949
...,...,...
2026-09-01,-0.014826,-0.007056
2026-09-02,0.006574,0.004251
2026-09-03,0.010532,0.010284


---
## 6. Result analysis

### 6.1 Return, volatility and Sharpe

In [1274]:
metrics_table =\
(
    pd.DataFrame(
        {column: compute_metrics(portfolio_returns[column], rf_daily, TRADING_DAYS)
         for column in portfolio_values.columns}
    )
    .rename_axis("Metric")
)

metrics_table

,Investor,Benchmark
Metric,,
Horizon Return,1.553094,1.964798
Annualised Return,0.136657,0.160119
Horizon Volatility,0.358003,0.529118
Annualised Volatility,0.132345,0.195601
Horizon Sharpe,2.197715,1.913369
Annualised Sharpe,0.812440,0.707324


### 6.2 Drawdowns and tail risk

Two views of downside: the **max drawdown** is the worst peak-to-trough loss the path ever sustained, a
peak-relative and path-dependent number; **CVaR** is the average loss on the worst
$1 - \text{confidence}$ of individual days, which ignores the path but says how heavy the left tail is.
One bounds the pain of holding on, the other the pain of a single bad day.

CVaR is reported twice: as a percentage, and as that percentage applied to the portfolio's **latest**
value. The dollar figure is what a worst-5% day costs on the balance actually held today, so it grows
with the account rather than staying pinned to `INITIAL_CAPITAL`.

In [1275]:
drawdowns =\
(
    portfolio_values
    .apply(compute_drawdown)
)

CVAR_LABEL        = f"CVaR ({CVAR_CONFIDENCE:.0%}, daily)"
CVAR_DOLLAR_LABEL = f"CVaR ({CVAR_CONFIDENCE:.0%}, daily $)"

latest_value =\
(
    portfolio_values
    .iloc[-1]                            # a Series per portfolio, so it aligns on the index
)

cvar =\
(
    portfolio_returns
    .apply(compute_cvar, confidence = CVAR_CONFIDENCE)
)

risk_table =\
(
    pd.DataFrame({"Max Drawdown"     : drawdowns.min(),
                  "Max Drawdown Date": drawdowns.idxmin(),
                  CVAR_LABEL         : cvar,
                  CVAR_DOLLAR_LABEL  : cvar.mul(latest_value)})   # same loss, on today's balance
    .rename_axis("Portfolio")
)

print("Investor portfolio worst loss from a prior peak: "
      f"{drawdowns['Investor'].min():.2%} on {drawdowns['Investor'].idxmin():%d %b %Y}")
print(f"Investor portfolio {CVAR_CONFIDENCE:.0%} CVaR: "
      f"{risk_table.loc['Investor', CVAR_LABEL]:.2%} — the average return on its worst "
      f"{(1 - CVAR_CONFIDENCE) * len(portfolio_returns):.0f} days, "
      f"or -${abs(risk_table.loc['Investor', CVAR_DOLLAR_LABEL]):,.0f} "
      f"on its latest value of ${latest_value['Investor']:,.0f}")

risk_table

Investor portfolio worst loss from a prior peak: -15.24% on 20 Mar 2020
Investor portfolio 95% CVaR: -2.02% — the average return on its worst 92 days, or -$5,160 on its latest value of $255,309


,Max Drawdown,Max Drawdown Date,"CVaR (95%, daily)","CVaR (95%, daily $)"
Portfolio,,,,
Investor,-0.152426,2020-03-20,-0.020209,"-5,159.557032"
Benchmark,-0.337173,2020-03-23,-0.029343,"-8,699.608986"


### 6.3 Alpha and beta

Investor excess return regressed on benchmark excess return.

The intercept is only interesting if it is distinguishable from zero, so the regression also tests

$$H_0:\ \alpha = 0 \qquad \text{against} \qquad H_1:\ \alpha \neq 0$$

Two standard errors are reported. The **OLS** one is the textbook formula, which assumes the
residuals are independent and identically distributed. Daily returns are neither — they are
volatility-clustered and mildly autocorrelated — so that error is typically too small and the
significance it implies too generous. The **Newey-West (HAC)** standard error corrects for both
heteroskedasticity and autocorrelation up to a truncation lag. Where the two disagree, the HAC
verdict is the one to believe.

In [1276]:
# === 1 day rebalancing ===
# R_p - R_f = 0.000199 + 2.0184 * (R_m - R_f)      [daily, R^2 = 0.8309]

# H_0: alpha = 0     (n = 1,425 daily observations)
#   OLS         SE = 0.000162   t =  1.226   p = 0.2202
#   Newey-West  SE = 0.000154   t =  1.298   p = 0.1946   [7 lags]

# At the 5% level we cannot reject H_0 on the HAC standard error.
# Annualised alpha 5.15%, 95% CI [-2.54%, 13.43%]


# === 60 day rebalancing ===
# R_p - R_f = 0.000203 + 2.0278 * (R_m - R_f)      [daily, R^2 = 0.8309]

# H_0: alpha = 0     (n = 1,425 daily observations)
#   OLS         SE = 0.000162   t =  1.248   p = 0.2122
#   Newey-West  SE = 0.000153   t =  1.324   p = 0.1858   [7 lags]

# At the 5% level we cannot reject H_0 on the HAC standard error.
# Annualised alpha 5.24%, 95% CI [-2.43%, 13.52%]


# === 100 day rebalancing ===
# R_p - R_f = 0.000209 + 2.0219 * (R_m - R_f)      [daily, R^2 = 0.8320]

# H_0: alpha = 0     (n = 1,425 daily observations)
#   OLS         SE = 0.000162   t =  1.292   p = 0.1967
#   Newey-West  SE = 0.000153   t =  1.365   p = 0.1726   [7 lags]

# At the 5% level we cannot reject H_0 on the HAC standard error.
# Annualised alpha 5.41%, 95% CI [-2.28%, 13.71%]



# === Without Rebalancing ===
# R_p - R_f = 0.000157 + 1.8483 * (R_m - R_f)      [daily, R^2 = 0.8523]

# H_0: alpha = 0     (n = 1,425 daily observations)
#   OLS         SE = 0.000152   t =  1.028   p = 0.3043
#   Newey-West  SE = 0.000145   t =  1.083   p = 0.2790   [7 lags]

# At the 5% level we cannot reject H_0 on the HAC standard error.
# Annualised alpha 4.03%, 95% CI [-3.15%, 11.73%]

In [1277]:
alpha_beta = compute_alpha_beta(excess_returns["Investor"],
                                excess_returns["Benchmark"],
                                TRADING_DAYS)

print(f"R_p - R_f = {alpha_beta['Alpha (daily)']:.6f} "
      f"+ {alpha_beta['Beta']:.4f} * (R_m - R_f)      [daily, R^2 = {alpha_beta['R-squared']:.4f}]")

print(f"\nH_0: alpha = 0     (n = {len(excess_returns):,} daily observations)")
print(f"  OLS         SE = {alpha_beta['SE Alpha (OLS)']:.6f}   "
      f"t = {alpha_beta['t-stat (OLS)']:6.3f}   p = {alpha_beta['p-value (OLS)']:.4f}")
print(f"  Newey-West  SE = {alpha_beta['SE Alpha (HAC)']:.6f}   "
      f"t = {alpha_beta['t-stat (HAC)']:6.3f}   p = {alpha_beta['p-value (HAC)']:.4f}   "
      f"[{alpha_beta['NW Lags']} lags]")

verdict = "reject" if alpha_beta["p-value (HAC)"] < 0.05 else "cannot reject"
print(f"\nAt the 5% level we {verdict} H_0 on the HAC standard error.")
print(f"Annualised alpha {alpha_beta['Annualised Alpha']:.2%}, "
      f"95% CI [{alpha_beta['Annual Alpha CI Low']:.2%}, "
      f"{alpha_beta['Annual Alpha CI High']:.2%}]")

pd.Series(alpha_beta).rename("Investor vs Benchmark").to_frame()

R_p - R_f = 0.000227 + 0.3639 * (R_m - R_f)      [daily, R^2 = 0.2893]

H_0: alpha = 0     (n = 1,844 daily observations)
  OLS         SE = 0.000164   t =  1.384   p = 0.1664
  Newey-West  SE = 0.000150   t =  1.517   p = 0.1296   [7 lags]

At the 5% level we cannot reject H_0 on the HAC standard error.
Annualised alpha 5.88%, 95% CI [-1.66%, 14.01%]


,Investor vs Benchmark
Beta,0.363880
Alpha (daily),0.000227
Annualised Alpha,0.058834
R-squared,0.289258
SE Alpha (OLS),0.000164
...,...
t-stat (HAC),1.516517
p-value (HAC),0.129560
NW Lags,7.000000
Annual Alpha CI Low,-0.016628


### 6.4 Summary table

Every computed figure in one place: returns, volatility, Sharpe, max drawdown and its date, CVaR,
then the whole regression — alpha and beta, both standard errors and their t-statistics and
p-values, the Newey-West truncation lag, and the annualised alpha confidence interval.

Alpha and beta describe the investor *relative to* the benchmark, so they are blank in the
benchmark's own column.

In [1278]:
summary_table =\
(
    metrics_table
    .T                                                       # portfolios -> rows
    # every risk column, then every key the regression returned; the regression rows are
    # investor-only, because the benchmark cannot have an alpha or a beta against itself
    .assign(**{metric: risk_table[metric] for metric in risk_table.columns},
            **{metric: [value, np.nan] for metric, value in alpha_beta.items()})
    .T                                                       # metrics -> rows
    .rename_axis("Metric")
)

summary_table

,Investor,Benchmark
Metric,,
Horizon Return,1.553094,1.964798
Annualised Return,0.136657,0.160119
Horizon Volatility,0.358003,0.529118
Annualised Volatility,0.132345,0.195601
Horizon Sharpe,2.197715,1.913369
...,...,...
t-stat (HAC),1.516517,NaN
p-value (HAC),0.129560,NaN
NW Lags,7.000000,NaN


In [1279]:
# Same table, formatted for reading. Each metric carries its own convention: rates as
# percentages, dollar losses with a sign in front of the currency symbol, daily alpha and its
# standard errors too small for a percentage, p-values that would round to a bare 0.0000, and
# the lag count as a plain integer.
percent_metrics = ["Horizon Return", "Annualised Return", "Horizon Volatility",
                   "Annualised Volatility", "Max Drawdown", CVAR_LABEL, "Annualised Alpha",
                   "Annual Alpha CI Low", "Annual Alpha CI High"]
dollar_metrics  = [CVAR_DOLLAR_LABEL]
daily_metrics   = ["Alpha (daily)", "SE Alpha (OLS)", "SE Alpha (HAC)"]
pvalue_metrics  = ["p-value (OLS)", "p-value (HAC)"]
count_metrics   = ["NW Lags"]


def format_metric(value: float, metric: str) -> str:
    """Render one cell of `summary_table`, choosing the convention from its metric name.

    Parameters
    ----------
    value  : float  the cell's value; `NaN` where the metric does not apply to that portfolio
    metric : str    the row label, i.e. which convention to use

    Returns
    -------
    str  the display string, "—" for a metric that does not apply
    """
    if pd.isna(value):
        return "—"

    if metric in percent_metrics:
        return f"{value:.2%}"
    if metric in dollar_metrics:
        return f"-${abs(value):,.0f}" if value < 0 else f"${value:,.0f}"
    if metric == "Max Drawdown Date":
        return f"{value:%d %b %Y}"
    if metric in daily_metrics:
        return f"{value:.6f}"
    if metric in pvalue_metrics:
        return "<0.0001" if value < 1e-4 else f"{value:.4f}"
    if metric in count_metrics:
        return f"{value:,.0f}"

    return f"{value:.4f}"


summary_display =\
(
    summary_table
    .apply(lambda row: row.map(lambda value: format_metric(value, row.name)),
           axis = "columns")
)

summary_display

,Investor,Benchmark
Metric,,
Horizon Return,155.31%,196.48%
Annualised Return,13.67%,16.01%
Horizon Volatility,35.80%,52.91%
Annualised Volatility,13.23%,19.56%
Horizon Sharpe,2.1977,1.9134
...,...,...
t-stat (HAC),1.5165,—
p-value (HAC),0.1296,—
NW Lags,7,—


---
## 7. Charts

In [1280]:
portfolio_values_long =\
(
    portfolio_values
    .reset_index()
    .melt(id_vars    = "Date",
          var_name   = "Portfolio",
          value_name = "Value")
)

portfolio_values_long

,Date,Portfolio,Value
0,2019-05-08,Investor,"100,000.000000"
1,2019-05-09,Investor,"99,918.724862"
2,2019-05-10,Investor,"100,028.769263"
3,2019-05-13,Investor,"98,972.288563"
4,2019-05-14,Investor,"99,455.960017"
...,...,...,...
3683,2026-09-01,Benchmark,"294,861.854867"
3684,2026-09-02,Benchmark,"296,170.128219"
3685,2026-09-03,Benchmark,"299,270.559113"
3686,2026-09-04,Benchmark,"298,117.099407"


In [1281]:
def describe(portfolio: dict[str, float]) -> str:
    """A weight dict as a one-line label, e.g. "SOXX 70% / GLD 30%".

    Parameters
    ----------
    portfolio : dict[str, float]  ticker -> target weight, as declared in §2

    Returns
    -------
    str  the holdings joined with " / ", each weight as a whole percentage
    """
    return " / ".join(f"{ticker} {weight:.0%}" for ticker, weight in portfolio.items())


# What the investor portfolio actually is, in one line. Under OPTIMISE_WEIGHTS the mix is re-solved
# at every rebalance, so a fixed "SOXX 70% / GLD 30%" label would misdescribe the backtest.
investor_label = (f"max-Sharpe MC over {', '.join(INVESTOR_PORTFOLIO)} "
                  f"({MC_RUNS:,} draws, {LOOKBACK_DAYS}d lookback)"
                  if OPTIMISE_WEIGHTS else
                  describe(INVESTOR_PORTFOLIO))


# Labels are derived from §2, not typed out, so they cannot drift from the weights actually
# backtested. `values` and `labels` are keyed by portfolio name rather than given as positional
# lists: lets-plot orders a discrete scale by first appearance, and `portfolio_values` puts
# Investor first, so positional lists hand each curve the other one's colour and label.
equity_plot =\
(
    ggplot(portfolio_values_long,
           aes(x = "Date",
               y = "Value")
          )
    + geom_line(aes(color = "Portfolio"),
                size = 0.8)
    + scale_color_manual(values = {"Benchmark": "blue",
                                   "Investor" : "red"},
                         labels = {"Benchmark": f"Benchmark: {describe(BENCHMARK_PORTFOLIO)}",
                                   "Investor" : f"Investor: {investor_label}"},
                         name   = "Portfolio")
    + scale_y_continuous(format = "$,.0f")
    + labs(title    = f"Growth of ${INITIAL_CAPITAL:,.0f}, rebalanced every {REBALANCE_DAYS} trading days",
           subtitle = f"{prices.index[0]:%d %b %Y} to {prices.index[-1]:%d %b %Y}",
           x        = "",
           y        = "Portfolio Value")
    + ggsize(1000, 500)
    + theme(legend_position = "top")
)

equity_plot

In [1282]:
drawdown_plot =\
(
    ggplot(drawdowns.reset_index()
                    .melt(id_vars    = "Date",
                          var_name   = "Portfolio",
                          value_name = "Drawdown"),
           aes(x = "Date",
               y = "Drawdown")
          )
    # Each portfolio is measured from 0, so the areas must overlap: lets-plot's default
    # position for geom_area is 'gstack', which would draw the second portfolio's fill on top
    # of the first and show a combined depth its own line never reaches.
    + geom_area(aes(fill = "Portfolio"),
                position = "identity",
                alpha    = 0.35)
    + geom_line(aes(color = "Portfolio"),
                size = 0.5)
    + scale_fill_manual(values  = {"Benchmark": "blue", "Investor": "red"}, name = "Portfolio")
    + scale_color_manual(values = {"Benchmark": "blue", "Investor": "red"}, name = "Portfolio")
    + scale_y_continuous(format = ".0%")
    + labs(title = "Underwater plot: decline from the running peak",
           x     = "",
           y     = "Drawdown")
    + ggsize(1000, 400)
    + theme(legend_position = "top")
)

drawdown_plot

In [1283]:
# How the investor's target mix moved over the horizon. These are the weights set *at* each
# rebalance, not the drifted weights mid-block: inside a block each sleeve compounds at its own
# rate, so the realised mix wanders away from the band drawn here until the next reset.
investor_weights_daily =\
(
    investor_weights
    .loc[np.arange(len(asset_returns)) // REBALANCE_DAYS]   # per-block weights -> one row per date
    .set_axis(asset_returns.index)
    .rename_axis("Date")
)

weights_plot =\
(
    ggplot(investor_weights_daily.reset_index()
                                 .melt(id_vars    = "Date",
                                       var_name   = "Ticker",
                                       value_name = "Weight"),
           aes(x = "Date",
               y = "Weight")
          )
    + geom_area(aes(fill = "Ticker"),          # weights sum to 1, so the stack fills 0-100%
                alpha = 0.75)
    + scale_y_continuous(format = ".0%",
                         limits = [0, 1])
    + labs(title    = "Investor target weights at each rebalance",
           subtitle = investor_label,
           x        = "",
           y        = "Weight")
    + ggsize(1000, 350)
    + theme(legend_position = "top")
)

weights_plot

In [1284]:
# Correlation of the daily returns of the assets the investor actually holds. Unlike covariance,
# correlation is already unitless and bounded to [-1, 1], so the colour scale can be anchored to
# that fixed range instead of the sample's own min/max — a -1 always reads as fully red here, not
# just "the most negative cell in this particular matrix". The diagonal is always 1 (an asset
# against itself). With a single-ticker INVESTOR_PORTFOLIO this collapses to one tile — it earns
# its keep once section 2 holds a mix, which is also when the optimiser in section 4 has something
# to solve.
investor_assets = list(INVESTOR_PORTFOLIO)

correlation =\
(
    asset_returns
    [investor_assets]
    .corr()
)

# lets-plot pads a degenerate continuous domain (every cell equal to 1, the single-ticker case)
# before it validates declared limits, so the fixed [-1, 1] range only applies once there is
# more than one tile to show.
correlation_limits = None if correlation.size == 1 else [-1, 1]

correlation_long =\
(
    correlation
    .rename_axis(index = "Row", columns = "Column")
    .stack()
    .rename("Correlation")
    .reset_index()
    # The number is the content of a matrix cell, so every cell carries one; the ink flips to
    # white only where the tile is dark enough to swallow black text.
    .assign(Label = lambda frame: frame["Correlation"].map("{:.2f}".format),
            Ink   = lambda frame: np.where(frame["Correlation"].abs() > 0.5, "white", "#0b0b0b"))
)

correlation_plot =\
(
    ggplot(correlation_long,
           aes(x = "Column",
               y = "Row")
          )
    + geom_tile(aes(fill = "Correlation"),
                width  = 0.96,               # a hairline of surface between cells
                height = 0.96)
    + geom_text(aes(label = "Label",
                    color = "Ink"),
                size = 9)
    # Positive and negative correlation are opposite meanings either side of "no linear
    # relationship" at zero, so the fill diverges from a neutral midpoint with equal arms —
    # fixed to [-1, 1] rather than the sample's range, so colour means the same thing across runs.
    + scale_fill_gradient2(low      = "#2a78d6",         # -1: moves opposite
                           mid      = "#f0efec",         #  0: no linear relationship
                           high     = "#e34948",         # +1: moves together
                           midpoint = 0,
                           limits   = correlation_limits,
                           name     = "Correlation")
    + scale_color_identity()                 # "Ink" holds the colour itself, not a category
    + scale_x_discrete(limits = investor_assets)
    + scale_y_discrete(limits = investor_assets[::-1])   # diagonal top-left -> bottom-right
    + coord_fixed()
    + labs(title    = "Correlation of daily returns, investor portfolio assets",
           subtitle = (f"{prices.index[0]:%d %b %Y} to {prices.index[-1]:%d %b %Y}  |  "
                       f"the diagonal is each asset against itself"),
           x        = "",
           y        = "")
    + ggsize(700, 600)
    # One tile carries no scale worth reading.
    + theme(legend_position = "none" if correlation.size == 1 else "right")
)

correlation_plot

In [1285]:
# The regression behind alpha and beta, drawn: each point is one trading day.
alpha_beta_plot =\
(
    ggplot(excess_returns.reset_index(),
           aes(x = "Benchmark",
               y = "Investor")
          )
    + geom_point(color = "grey",
                 alpha = 0.20,
                 size  = 1.5)
    + geom_abline(slope     = alpha_beta["Beta"],
                  intercept = alpha_beta["Alpha (daily)"],
                  color     = "red",
                  size      = 1.0)
    + geom_hline(yintercept = 0, color = "black", size = 0.3)
    + geom_vline(xintercept = 0, color = "black", size = 0.3)
    + scale_x_continuous(format = ".1%")
    + scale_y_continuous(format = ".1%")
    + labs(title    = (f"R_p - R_f  =  {alpha_beta['Alpha (daily)']:.6f}  "
                       f"+  {alpha_beta['Beta']:.4f} (R_m - R_f)"),
           subtitle = (f"Daily excess returns  |  R-squared = {alpha_beta['R-squared']:.4f}  |  "
                       f"annualised alpha = {alpha_beta['Annualised Alpha']:.2%}  |  "
                       f"Newey-West t = {alpha_beta['t-stat (HAC)']:.2f}, p = {alpha_beta['p-value (HAC)']:.4f}"),
           x        = "Benchmark excess return  (R_m - R_f)",
           y        = "Investor excess return  (R_p - R_f)")
    + ggsize(700, 700)
)

alpha_beta_plot

In [1286]:
dashboard =\
(
    gggrid([equity_plot, correlation_plot, drawdown_plot, weights_plot],
           ncol = 2)
    + ggsize(1250, 1000)
)

dashboard

## 8. At a glance

In [1289]:
# The whole result in one output. Change the weights or flip OPTIMISE_WEIGHTS in §2, run all,
# then read only this cell:
# every figure and label below is derived, so nothing here ever needs editing.
verdict = "reject" if alpha_beta["p-value (HAC)"] < 0.05 else "cannot reject"

# Escape the currency where it is literally currency — the table and the initial capital — so
# MathJax skips it and reads only the $$...$$ block as maths. Escaping the whole string, as an
# earlier version did, would blank the formula by escaping its delimiters too.
summary_table_md = summary_display.to_markdown().replace("$", r"\$")
initial_capital  = rf"\${INITIAL_CAPITAL:,.0f}"

summary_markdown = rf"""
---

### **Investor** {investor_label} &nbsp; vs &nbsp; **Benchmark** {describe(BENCHMARK_PORTFOLIO)}

{prices.index[0]:%d %b %Y} → {prices.index[-1]:%d %b %Y} · {HORIZON_DAYS:,} trading days · {initial_capital} initial · rebalanced every {REBALANCE_DAYS} trading days

{summary_table_md}

$$
R_p - R_f \;=\; {alpha_beta['Alpha (daily)']:.6f} \;+\; {alpha_beta['Beta']:.4f}\,(R_m - R_f)
\qquad \text{{daily}},\; R^2 = {alpha_beta['R-squared']:.4f}
$$

#### At the 5% level we **{verdict}** H₀: α = 0 on the Newey-West standard error (p = {alpha_beta['p-value (HAC)']:.4f}).
"""

display(Markdown(summary_markdown))
display(dashboard)


---

### **Investor** max-Sharpe MC over QQQ, DBMF, GLD (10,000 draws, 1260d lookback) &nbsp; vs &nbsp; **Benchmark** SPY 100%

08 May 2019 → 08 Sep 2026 · 1,844 trading days · \$100,000 initial · rebalanced every 60 trading days

| Metric                | Investor    | Benchmark   |
|:----------------------|:------------|:------------|
| Horizon Return        | 155.31%     | 196.48%     |
| Annualised Return     | 13.67%      | 16.01%      |
| Horizon Volatility    | 35.80%      | 52.91%      |
| Annualised Volatility | 13.23%      | 19.56%      |
| Horizon Sharpe        | 2.1977      | 1.9134      |
| Annualised Sharpe     | 0.8124      | 0.7073      |
| Max Drawdown          | -15.24%     | -33.72%     |
| Max Drawdown Date     | 20 Mar 2020 | 23 Mar 2020 |
| CVaR (95%, daily)     | -2.02%      | -2.93%      |
| CVaR (95%, daily \$)   | -\$5,160     | -\$8,700     |
| Beta                  | 0.3639      | —           |
| Alpha (daily)         | 0.000227    | —           |
| Annualised Alpha      | 5.88%       | —           |
| R-squared             | 0.2893      | —           |
| SE Alpha (OLS)        | 0.000164    | —           |
| t-stat (OLS)          | 1.3845      | —           |
| p-value (OLS)         | 0.1664      | —           |
| SE Alpha (HAC)        | 0.000150    | —           |
| t-stat (HAC)          | 1.5165      | —           |
| p-value (HAC)         | 0.1296      | —           |
| NW Lags               | 7           | —           |
| Annual Alpha CI Low   | -1.66%      | —           |
| Annual Alpha CI High  | 14.01%      | —           |

$$
R_p - R_f \;=\; 0.000227 \;+\; 0.3639\,(R_m - R_f)
\qquad \text{daily},\; R^2 = 0.2893
$$

#### At the 5% level we **cannot reject** H₀: α = 0 on the Newey-West standard error (p = 0.1296).
